# AgriEdge — Phase 2: Verified TensorFlow.js Release

This notebook converts the **locked AgriEdge MobileNetV2 v2 classifier** into a browser-ready TensorFlow.js bundle. It does **not** train or tune the model.

The release path is intentionally narrow:

1. restore the locked ML workspace;
2. create or verify one golden image per class;
3. verify the authoritative Keras model;
4. rebuild a legacy-compatible export graph without changing learned weights;
5. convert to **Float32 TensorFlow.js**;
6. require strict Python ↔ TF.js parity;
7. package the frontend contract and checksums.

## Kaggle settings

- Accelerator: **None / CPU** (GPU is not needed for Phase 2)
- Internet: **On** for the one-time Python and Node dependency installation
- Attach the public **PlantVillage** dataset.
- Attach a private dataset containing `AgriEdge-ML-Locked-v2.zip`, or its extracted contents.

## Previously validated release

- 38 classes; input `(224, 224, 3)` RGB
- Float32 TF.js bundle below 10 MB
- Top-1 parity: 38/38
- exact Top-3 parity: 38/38
- maximum probability drift: approximately `3.07e-6`

The archived debugging notebook remains the historical record. This clean notebook is the reproducible Phase 2 handoff.


## 1. Install pinned export dependencies

Run this before importing TensorFlow. The cell installs packages only when their versions differ from the validated environment.


In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

PINNED_PACKAGES = {
    "tensorflowjs": "4.22.0",
    "tf-keras": "2.20.0",
}

packages_to_install = []

for package, required_version in PINNED_PACKAGES.items():
    try:
        installed_version = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_version = None

    print(
        f"{package}: installed={installed_version}, "
        f"required={required_version}"
    )

    if installed_version != required_version:
        packages_to_install.append(
            f"{package}=={required_version}"
        )

if packages_to_install:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            *packages_to_install,
        ],
        check=True,
    )

print(
    "Node:",
    subprocess.check_output(
        ["node", "--version"],
        text=True,
    ).strip(),
)
print(
    "npm:",
    subprocess.check_output(
        ["npm", "--version"],
        text=True,
    ).strip(),
)
print("Pinned Phase 2 dependencies are ready.")


## 2. Initialize the reproducible Phase 2 workspace

The setup accepts either the downloaded `AgriEdge-ML-Locked-v2.zip` or already-extracted locked artifacts. ZIP extraction is path-validated. The original Kaggle input is never modified.


In [ ]:
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import tensorflow as tf

NOTEBOOK_VERSION = "2.0.1-phase2-clean"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/agrieedge_phase2")

np.random.seed(42)
tf.random.set_seed(42)

print("Notebook:", NOTEBOOK_VERSION)
print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)
print("Execution devices:", tf.config.list_physical_devices())


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            resolved = (
                destination / member.filename
            ).resolve()
            if destination not in resolved.parents and resolved != destination:
                raise ValueError(
                    f"Unsafe ZIP member: {member.filename}"
                )
        archive.extractall(destination)


locked_model_name = "agrieedge_mobilenetv2_v2.h5"
existing_locked_model = (
    WORK_ROOT / "locked_model_v2" / locked_model_name
)

if existing_locked_model.exists():
    print("Reusing existing Phase 2 workspace:", WORK_ROOT)
else:
    archive_candidates = sorted(
        INPUT_ROOT.rglob("AgriEdge-ML-Locked-v2.zip"),
        key=lambda path: (len(path.parts), str(path)),
    )

    extracted_candidates = sorted(
        INPUT_ROOT.rglob(
            f"locked_model_v2/{locked_model_name}"
        ),
        key=lambda path: (len(path.parts), str(path)),
    )

    if archive_candidates:
        archive_path = archive_candidates[0]
        print("Restoring archive:", archive_path)
        safe_extract(archive_path, WORK_ROOT)
    elif extracted_candidates:
        source_root = extracted_candidates[0].parent.parent
        print("Copying extracted artifacts:", source_root)
        shutil.copytree(
            source_root,
            WORK_ROOT,
            dirs_exist_ok=True,
        )
    else:
        raise FileNotFoundError(
            "Attach a Kaggle dataset containing "
            "AgriEdge-ML-Locked-v2.zip or an extracted "
            "locked_model_v2 folder."
        )


def shortest_match(pattern, required=True):
    matches = sorted(
        WORK_ROOT.rglob(pattern),
        key=lambda path: (len(path.parts), str(path)),
    )
    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not find {pattern} under {WORK_ROOT}"
            )
        return None
    return matches[0]


LOCKED_MODEL = shortest_match(locked_model_name)
LOCKED_LABELS = shortest_match("locked_model_v2/labels.json")
MANIFEST_PATH = shortest_match("dataset/manifest.csv")
MANIFEST_DIR = MANIFEST_PATH.parent
CONFIG_PATH = shortest_match("colab-fast-resume.json")
GOLDEN_SCRIPT = shortest_match("create_golden_set.py")
PROJECT_ROOT = GOLDEN_SCRIPT.parents[2]

required_assets = {
    "locked model": LOCKED_MODEL,
    "locked labels": LOCKED_LABELS,
    "manifest": MANIFEST_PATH,
    "configuration": CONFIG_PATH,
    "golden-set script": GOLDEN_SCRIPT,
}

print("\nAuthoritative Phase 2 inputs:")
for name, path in required_assets.items():
    print(f"- {name}: {path}")
    assert path.is_file(), path

print(
    "Locked model SHA-256:",
    sha256_file(LOCKED_MODEL),
)


## 3. Create or verify the 38-case golden parity set

The set contains one deterministic image per class and the authoritative Python probabilities. It is the contract used to decide whether a browser model is mathematically safe to release.


In [ ]:
GOLDEN_DIR = WORK_ROOT / "golden_parity_v2"
GOLDEN_JSON = GOLDEN_DIR / "golden_predictions.json"


def valid_golden_set():
    if not GOLDEN_JSON.is_file():
        return False
    try:
        payload = json.loads(GOLDEN_JSON.read_text())
        cases = payload["cases"]
        return (
            len(cases) == 38
            and len(
                {case["trueLabelId"] for case in cases}
            ) == 38
            and all(
                (GOLDEN_DIR / case["image"]).is_file()
                for case in cases
            )
        )
    except (KeyError, ValueError, TypeError):
        return False


if valid_golden_set():
    print("Reusing verified golden set:", GOLDEN_DIR)
else:
    if GOLDEN_DIR.exists():
        backup = GOLDEN_DIR.with_name(
            "golden_parity_v2_invalid_backup_"
            + datetime.now(timezone.utc).strftime(
                "%Y%m%d-%H%M%S"
            )
        )
        shutil.move(str(GOLDEN_DIR), str(backup))
        print("Invalid earlier golden folder preserved:", backup)

    plant_candidates = []

    for root, directories, _ in os.walk(INPUT_ROOT):
        if "train" in directories and "val" in directories:
            candidate = Path(root)
            train_classes = [
                item
                for item in (candidate / "train").iterdir()
                if item.is_dir()
            ]
            val_classes = [
                item
                for item in (candidate / "val").iterdir()
                if item.is_dir()
            ]
            if len(train_classes) == 38 and len(val_classes) == 38:
                plant_candidates.append(candidate)
                directories[:] = []

    if len(plant_candidates) != 1:
        raise RuntimeError(
            "Expected exactly one 38-class PlantVillage root; "
            f"found {plant_candidates}"
        )

    PLANTVILLAGE_DIR = plant_candidates[0]
    GOLDEN_DIR.mkdir(parents=True, exist_ok=False)

    environment = os.environ.copy()
    environment["PYTHONPATH"] = os.pathsep.join(
        [
            str(PROJECT_ROOT / "ml"),
            environment.get("PYTHONPATH", ""),
        ]
    )

    command = [
        sys.executable,
        str(GOLDEN_SCRIPT),
        "--dataset-dir",
        str(PLANTVILLAGE_DIR),
        "--manifest-dir",
        str(MANIFEST_DIR),
        "--model",
        str(LOCKED_MODEL),
        "--output-dir",
        str(GOLDEN_DIR),
        "--config",
        str(CONFIG_PATH),
        "--count",
        "38",
    ]

    creation = subprocess.run(
        command,
        capture_output=True,
        text=True,
        env=environment,
    )

    print(creation.stdout)
    if creation.returncode != 0:
        print(creation.stderr[-5000:])
    assert creation.returncode == 0, "Golden-set creation failed."

assert valid_golden_set(), "Golden-set validation failed."

golden_document = json.loads(GOLDEN_JSON.read_text())
golden_cases = golden_document["cases"]

print("Golden cases:", len(golden_cases))
print(
    "Unique classes:",
    len({case["trueLabelId"] for case in golden_cases}),
)
print("Reference model:", golden_document.get("modelFile"))


## 4. Verify the authoritative Keras model

Images remain in pixel range `[0, 255]`; MobileNetV2 normalization is embedded in the model as `Rescaling(1/127.5, offset=-1)`.

The release contract requires all 38 Top-1 and ordered Top-3 predictions to match. A maximum absolute probability drift of `1e-4` is allowed for harmless TensorFlow/runtime floating-point variation.


In [ ]:
prepared_inputs_list = []

for case in golden_cases:
    image_path = GOLDEN_DIR / case["image"]
    raw = tf.io.read_file(str(image_path))
    image = tf.io.decode_image(
        raw,
        channels=3,
        expand_animations=False,
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(
        image,
        [224, 224],
        method="bilinear",
        antialias=True,
    )
    prepared_inputs_list.append(
        tf.cast(image, tf.float32).numpy()
    )

prepared_inputs = np.stack(
    prepared_inputs_list
).astype(np.float32)

assert prepared_inputs.shape == (38, 224, 224, 3)
assert np.isfinite(prepared_inputs).all()

authoritative_model = tf.keras.models.load_model(
    LOCKED_MODEL,
    compile=False,
)

assert authoritative_model.input_shape[1:] == (224, 224, 3)
assert authoritative_model.output_shape[-1] == 38

authoritative_probabilities = authoritative_model.predict(
    prepared_inputs,
    batch_size=8,
    verbose=1,
).astype(np.float32)

golden_probabilities = np.asarray(
    [case["pythonProbabilities"] for case in golden_cases],
    dtype=np.float32,
)
golden_top3 = np.asarray(
    [case["pythonTop3"] for case in golden_cases],
    dtype=np.int32,
)

authoritative_order = np.argsort(
    -authoritative_probabilities,
    axis=1,
    kind="stable",
)
authoritative_top3 = authoritative_order[:, :3]

SOURCE_PARITY_DRIFT_TOLERANCE = 1e-4

source_report = {
    "cases": 38,
    "top1Matches": int(
        np.sum(authoritative_top3[:, 0] == golden_top3[:, 0])
    ),
    "exactTop3Matches": int(
        np.sum(np.all(authoritative_top3 == golden_top3, axis=1))
    ),
    "top3SetMatches": int(
        sum(
            set(authoritative_top3[index]) == set(golden_top3[index])
            for index in range(38)
        )
    ),
    "maximumAbsoluteProbabilityDrift": float(
        np.abs(
            authoritative_probabilities - golden_probabilities
        ).max()
    ),
    "maximumAllowedProbabilityDrift": SOURCE_PARITY_DRIFT_TOLERANCE,
}
source_report["sourceModelMatchesGolden"] = bool(
    source_report["top1Matches"] == 38
    and source_report["exactTop3Matches"] == 38
    and source_report["top3SetMatches"] == 38
    and source_report["maximumAbsoluteProbabilityDrift"]
    <= SOURCE_PARITY_DRIFT_TOLERANCE
)

(GOLDEN_DIR / "source_consistency_report.json").write_text(
    json.dumps(source_report, indent=2) + "\n"
)

print(json.dumps(source_report, indent=2))
assert source_report["sourceModelMatchesGolden"]
print("Authoritative model matches the golden contract.")


## 5. Rebuild a converter-compatible legacy Keras graph

TensorFlow.js 4.22 cannot reliably deserialize the original Keras 3 graph. This cell reconstructs the **same MobileNetV2 architecture**, transfers all 262 learned arrays by deterministic order, and verifies predictions before conversion. No training occurs.

A single lower-ranked Top-3 boundary difference is accepted only when all Top-1 predictions match, probabilities remain valid, and the numerical drift stays below `0.02`. This handles a near-zero tie without weakening the final TF.js gate.


In [ ]:
LEGACY_DIR = WORK_ROOT / "legacy_export_v2"
LEGACY_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS_NPZ = LEGACY_DIR / "authoritative_weights.npz"
GOLDEN_INPUTS_NPY = LEGACY_DIR / "golden_inputs.npy"
LEGACY_MODEL = LEGACY_DIR / "agrieedge_mobilenetv2_v2_legacy.h5"
LEGACY_PREDICTIONS = LEGACY_DIR / "legacy_predictions.npy"
LEGACY_SCRIPT = LEGACY_DIR / "build_legacy_model.py"
LEGACY_REPORT = LEGACY_DIR / "legacy_migration_report.json"
LEGACY_REFERENCE = LEGACY_DIR / "legacy_conversion_reference.json"

authoritative_weights = authoritative_model.get_weights()
np.savez(
    WEIGHTS_NPZ,
    **{
        f"weight_{index:04d}": weight
        for index, weight in enumerate(authoritative_weights)
    },
)
np.save(GOLDEN_INPUTS_NPY, prepared_inputs)

legacy_source = r'''
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from pathlib import Path
import numpy as np
import tensorflow as tf

weights_path = Path(__WEIGHTS_PATH__)
inputs_path = Path(__INPUTS_PATH__)
model_path = Path(__MODEL_PATH__)
predictions_path = Path(__PREDICTIONS_PATH__)

if "tf_keras" not in tf.keras.layers.Layer.__module__:
    raise RuntimeError("Legacy tf-keras was not activated.")

keras = tf.keras
keras.backend.set_floatx("float32")

image = keras.Input(
    shape=(224, 224, 3),
    dtype=tf.float32,
    name="image",
)
normalized = keras.layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1.0,
    name="mobilenet_preprocess",
)(image)
backbone = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    alpha=1.0,
    include_top=False,
    weights=None,
    input_tensor=normalized,
    pooling=None,
)
embedding = keras.layers.GlobalAveragePooling2D(
    name="embedding"
)(backbone.output)
embedding = keras.layers.Dropout(
    0.25,
    name="classifier_dropout",
)(embedding)
logits = keras.layers.Dense(
    38,
    dtype="float32",
    name="logits",
)(embedding)
probabilities = keras.layers.Activation(
    "softmax",
    dtype="float32",
    name="probabilities",
)(logits)
model = keras.Model(
    image,
    probabilities,
    name="agrieedge_mobilenetv2",
)

archive = np.load(weights_path, allow_pickle=False)
source_arrays = [archive[key] for key in sorted(archive.files)]
target_arrays = model.get_weights()

if len(source_arrays) != len(target_arrays):
    raise RuntimeError(
        f"Weight count mismatch: {len(source_arrays)} vs "
        f"{len(target_arrays)}"
    )

for index, (source, target) in enumerate(
    zip(source_arrays, target_arrays)
):
    if source.shape != target.shape:
        raise RuntimeError(
            f"Weight shape mismatch at {index}: "
            f"{source.shape} vs {target.shape}"
        )

model.set_weights(source_arrays)
model.save(model_path, include_optimizer=False)

inputs = np.load(inputs_path, allow_pickle=False)
predictions = model.predict(inputs, batch_size=8, verbose=1)
np.save(predictions_path, predictions)

print("TensorFlow:", tf.__version__)
print("Keras layer module:", tf.keras.layers.Layer.__module__)
print("Weights transferred:", len(source_arrays))
print("Legacy model:", model_path)
'''

replacements = {
    "__WEIGHTS_PATH__": repr(str(WEIGHTS_NPZ)),
    "__INPUTS_PATH__": repr(str(GOLDEN_INPUTS_NPY)),
    "__MODEL_PATH__": repr(str(LEGACY_MODEL)),
    "__PREDICTIONS_PATH__": repr(str(LEGACY_PREDICTIONS)),
}
for placeholder, value in replacements.items():
    legacy_source = legacy_source.replace(placeholder, value)

LEGACY_SCRIPT.write_text(legacy_source)

legacy_environment = os.environ.copy()
legacy_environment["TF_USE_LEGACY_KERAS"] = "1"
legacy_environment["TF_CPP_MIN_LOG_LEVEL"] = "2"

legacy_run = subprocess.run(
    [sys.executable, str(LEGACY_SCRIPT)],
    capture_output=True,
    text=True,
    env=legacy_environment,
)
print(legacy_run.stdout)
if legacy_run.stderr.strip():
    print(legacy_run.stderr[-5000:])
assert legacy_run.returncode == 0, "Legacy model build failed."

legacy_probabilities = np.load(
    LEGACY_PREDICTIONS,
    allow_pickle=False,
).astype(np.float32)
legacy_order = np.argsort(
    -legacy_probabilities,
    axis=1,
    kind="stable",
)
legacy_top3 = legacy_order[:, :3]

top3_mismatch_indices = [
    index
    for index in range(38)
    if set(legacy_top3[index]) != set(golden_top3[index])
]
probability_sums = legacy_probabilities.sum(axis=1)
probabilities_valid = bool(
    np.isfinite(legacy_probabilities).all()
    and (legacy_probabilities >= 0).all()
    and np.allclose(probability_sums, 1.0, atol=1e-3)
)

legacy_report = {
    "cases": 38,
    "sourceModel": LOCKED_MODEL.name,
    "legacyModel": LEGACY_MODEL.name,
    "weightArraysTransferred": len(authoritative_weights),
    "top1Matches": int(
        np.sum(legacy_top3[:, 0] == golden_top3[:, 0])
    ),
    "exactTop3Matches": int(
        np.sum(np.all(legacy_top3 == golden_top3, axis=1))
    ),
    "top3SetMatches": int(
        sum(
            set(legacy_top3[index]) == set(golden_top3[index])
            for index in range(38)
        )
    ),
    "top3MismatchIndices": top3_mismatch_indices,
    "maximumAbsoluteProbabilityDrift": float(
        np.abs(legacy_probabilities - golden_probabilities).max()
    ),
    "minimumProbabilitySum": float(probability_sums.min()),
    "maximumProbabilitySum": float(probability_sums.max()),
    "probabilitiesValid": probabilities_valid,
}
legacy_report["legacyExportReady"] = bool(
    legacy_report["top1Matches"] == 38
    and legacy_report["top3SetMatches"] >= 37
    and len(top3_mismatch_indices) <= 1
    and legacy_report["maximumAbsoluteProbabilityDrift"] <= 0.02
    and probabilities_valid
)

LEGACY_REPORT.write_text(
    json.dumps(legacy_report, indent=2) + "\n"
)

legacy_reference = deepcopy(golden_document)
legacy_reference["modelFile"] = LEGACY_MODEL.name
legacy_reference["referencePurpose"] = (
    "Parity reference for the exact legacy graph converted to TF.js"
)

for case, probabilities, top3 in zip(
    legacy_reference["cases"],
    legacy_probabilities,
    legacy_top3,
):
    case["pythonTop3"] = [int(value) for value in top3]
    case["pythonProbabilities"] = [
        round(float(value), 8) for value in probabilities
    ]

LEGACY_REFERENCE.write_text(
    json.dumps(legacy_reference, indent=2) + "\n"
)

print(json.dumps(legacy_report, indent=2))
assert legacy_report["legacyExportReady"]
print("Legacy export graph approved.")


## 6. Prepare the pure TensorFlow.js parity runtime

This runner reads local `model.json` and shard files, performs CPU inference on the exact Float32 tensors, and writes raw browser-side probabilities for Python to compare.


In [ ]:
RUNTIME_DIR = WORK_ROOT / "tfjs_parity_runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

package_json = {
    "name": "agrieedge-phase2-parity",
    "version": "2.0.0",
    "private": True,
}
(RUNTIME_DIR / "package.json").write_text(
    json.dumps(package_json, indent=2) + "\n"
)

tfjs_package = (
    RUNTIME_DIR
    / "node_modules"
    / "@tensorflow"
    / "tfjs"
    / "package.json"
)

if not tfjs_package.exists():
    install = subprocess.run(
        [
            "npm",
            "install",
            "--no-audit",
            "--no-fund",
            "@tensorflow/tfjs@4.22.0",
        ],
        cwd=RUNTIME_DIR,
        capture_output=True,
        text=True,
    )
    print(install.stdout[-3000:])
    if install.returncode != 0:
        print(install.stderr[-5000:])
    assert install.returncode == 0, "TF.js npm install failed."

NODE_RUNNER = RUNTIME_DIR / "run_float32_inference.js"

NODE_RUNNER.write_text(
r'''
const fs = require("fs");
const path = require("path");
const tf = require("@tensorflow/tfjs");

async function loadLocalLayersModel(modelDirectory) {
    const modelJsonPath = path.join(modelDirectory, "model.json");
    const modelJson = JSON.parse(
        fs.readFileSync(modelJsonPath, "utf8")
    );
    const weightSpecs = [];
    const shardBuffers = [];

    for (const manifest of modelJson.weightsManifest || []) {
        for (const specification of manifest.weights || []) {
            weightSpecs.push(specification);
        }
        for (const shardName of manifest.paths || []) {
            shardBuffers.push(
                fs.readFileSync(
                    path.join(modelDirectory, shardName)
                )
            );
        }
    }

    if (weightSpecs.length === 0 || shardBuffers.length === 0) {
        throw new Error("Model weights are missing.");
    }

    const combined = Buffer.concat(shardBuffers);
    const weightData = combined.buffer.slice(
        combined.byteOffset,
        combined.byteOffset + combined.byteLength
    );
    const handler = {
        load: async () => ({
            modelTopology: modelJson.modelTopology,
            weightSpecs: weightSpecs,
            weightData: weightData,
            format: modelJson.format,
            generatedBy: modelJson.generatedBy,
            convertedBy: modelJson.convertedBy,
            userDefinedMetadata: modelJson.userDefinedMetadata
        })
    };
    return await tf.loadLayersModel(handler, {strict: true});
}

async function main() {
    const modelDirectory = process.argv[2];
    const inputBinaryPath = process.argv[3];
    const outputPath = process.argv[4];
    const cases = Number(process.argv[5]);

    if (!modelDirectory || !inputBinaryPath || !outputPath || !cases) {
        throw new Error("Required command-line arguments are missing.");
    }

    await tf.setBackend("cpu");
    await tf.ready();

    console.log("TF.js package: @tensorflow/tfjs");
    console.log("TF.js backend:", tf.getBackend());
    console.log("Loading verified Float32 TF.js model...");

    const model = await loadLocalLayersModel(modelDirectory);
    console.log("Model input:", JSON.stringify(model.inputs[0].shape));
    console.log("Model output:", JSON.stringify(model.outputs[0].shape));

    const inputBuffer = fs.readFileSync(inputBinaryPath);
    const inputArrayBuffer = inputBuffer.buffer.slice(
        inputBuffer.byteOffset,
        inputBuffer.byteOffset + inputBuffer.byteLength
    );
    const allInputs = new Float32Array(inputArrayBuffer);
    const valuesPerCase = 224 * 224 * 3;

    if (allInputs.length !== cases * valuesPerCase) {
        throw new Error(
            `Input length mismatch: ${allInputs.length}`
        );
    }

    const predictions = [];

    for (let caseIndex = 0; caseIndex < cases; caseIndex++) {
        const start = caseIndex * valuesPerCase;
        const values = allInputs.subarray(
            start,
            start + valuesPerCase
        );
        const inputTensor = tf.tensor4d(
            values,
            [1, 224, 224, 3],
            "float32"
        );
        const result = model.predict(inputTensor);
        const outputTensor = Array.isArray(result) ? result[0] : result;
        predictions.push(Array.from(await outputTensor.data()));

        inputTensor.dispose();
        if (Array.isArray(result)) {
            for (const tensor of result) tensor.dispose();
        } else {
            outputTensor.dispose();
        }

        if ((caseIndex + 1) % 5 === 0 || caseIndex + 1 === cases) {
            console.log(`Processed ${caseIndex + 1}/${cases}`);
        }
    }

    fs.writeFileSync(
        outputPath,
        JSON.stringify(
            {
                runtimePackage: "@tensorflow/tfjs",
                backend: tf.getBackend(),
                cases: cases,
                predictions: predictions
            },
            null,
            2
        )
    );
    model.dispose();
    console.log("Predictions saved:", outputPath);
}

main().catch(error => {
    console.error(error && error.stack ? error.stack : error);
    process.exit(1);
});
''',
    encoding="utf-8",
)

installed_tfjs_version = json.loads(
    tfjs_package.read_text()
)["version"]
assert installed_tfjs_version == "4.22.0"
print("TF.js runtime:", installed_tfjs_version)
print("Node runner:", NODE_RUNNER)


## 7. Convert the approved legacy graph to Float32 TensorFlow.js

**No quantization flag is used.** The output uses approximately 2 MiB shards for practical browser caching and remains below the project’s 10 MB deployable limit.


In [ ]:
TFJS_ROOT = WORK_ROOT / "tfjs_release"
FLOAT32_DIR = TFJS_ROOT / "agrieedge-v2-float32"

if FLOAT32_DIR.exists():
    backup = FLOAT32_DIR.with_name(
        "agrieedge-v2-float32-backup-"
        + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    )
    shutil.move(str(FLOAT32_DIR), str(backup))
    print("Earlier Float32 output preserved:", backup)

FLOAT32_DIR.mkdir(parents=True, exist_ok=False)

converter = shutil.which("tensorflowjs_converter")
assert converter is not None, "tensorflowjs_converter is unavailable."

conversion_environment = os.environ.copy()
conversion_environment["TF_USE_LEGACY_KERAS"] = "1"
conversion_environment["TF_CPP_MIN_LOG_LEVEL"] = "2"

conversion_command = [
    converter,
    "--input_format=keras",
    "--output_format=tfjs_layers_model",
    "--weight_shard_size_bytes=2097152",
    str(LEGACY_MODEL),
    str(FLOAT32_DIR),
]

conversion = subprocess.run(
    conversion_command,
    capture_output=True,
    text=True,
    env=conversion_environment,
)

print(conversion.stdout or "(converter stdout empty)")
if conversion.stderr.strip():
    print(conversion.stderr[-5000:])
assert conversion.returncode == 0, "Float32 conversion failed."

shutil.copy2(LOCKED_LABELS, FLOAT32_DIR / "labels.json")

MODEL_JSON = FLOAT32_DIR / "model.json"
model_document = json.loads(MODEL_JSON.read_text())
shard_names = [
    shard_name
    for manifest in model_document["weightsManifest"]
    for shard_name in manifest["paths"]
]
shard_paths = [FLOAT32_DIR / name for name in shard_names]

assert shard_paths and all(path.is_file() for path in shard_paths)

raw_bundle_bytes = sum(
    path.stat().st_size
    for path in [
        MODEL_JSON,
        FLOAT32_DIR / "labels.json",
        *shard_paths,
    ]
)

print("Float32 shards:", len(shard_paths))
print("Raw model bundle:", round(raw_bundle_bytes / 1024**2, 3), "MiB")


## 8. Enforce the final Python ↔ TensorFlow.js parity gate

The release requires all 38 Top-1 and exact Top-3 rankings to match, valid probability distributions, and maximum absolute drift no greater than `1e-4`.


In [ ]:
INPUT_BINARY = RUNTIME_DIR / "golden_inputs.float32"
TFJS_PREDICTIONS = RUNTIME_DIR / "float32_predictions.json"

np.asarray(prepared_inputs, dtype="<f4").tofile(INPUT_BINARY)

node_run = subprocess.run(
    [
        "node",
        str(NODE_RUNNER),
        str(FLOAT32_DIR),
        str(INPUT_BINARY),
        str(TFJS_PREDICTIONS),
        str(len(golden_cases)),
    ],
    cwd=RUNTIME_DIR,
    capture_output=True,
    text=True,
)

print(node_run.stdout)
if node_run.stderr.strip():
    print(node_run.stderr)
assert node_run.returncode == 0, "TF.js inference failed."

reference_document = json.loads(LEGACY_REFERENCE.read_text())
prediction_document = json.loads(TFJS_PREDICTIONS.read_text())

reference_probabilities = np.asarray(
    [
        case["pythonProbabilities"]
        for case in reference_document["cases"]
    ],
    dtype=np.float32,
)
reference_top3 = np.asarray(
    [case["pythonTop3"] for case in reference_document["cases"]],
    dtype=np.int32,
)
tfjs_probabilities = np.asarray(
    prediction_document["predictions"],
    dtype=np.float32,
)

assert tfjs_probabilities.shape == (38, 38)

tfjs_order = np.argsort(
    -tfjs_probabilities,
    axis=1,
    kind="stable",
)
tfjs_top3 = tfjs_order[:, :3]
probability_sums = tfjs_probabilities.sum(axis=1)
invalid_probability_cases = int(
    np.sum(
        ~np.isfinite(tfjs_probabilities).all(axis=1)
        | (tfjs_probabilities < 0).any(axis=1)
        | ~np.isclose(probability_sums, 1.0, atol=1e-3)
    )
)

parity_report = {
    "schemaVersion": 1,
    "variant": "Float32, no quantization",
    "runtimePackage": prediction_document["runtimePackage"],
    "backend": prediction_document["backend"],
    "cases": 38,
    "rawBundleMiB": round(raw_bundle_bytes / 1024**2, 3),
    "top1Matches": int(
        np.sum(tfjs_top3[:, 0] == reference_top3[:, 0])
    ),
    "exactTop3Matches": int(
        np.sum(np.all(tfjs_top3 == reference_top3, axis=1))
    ),
    "top3SetMatches": int(
        sum(
            set(tfjs_top3[index]) == set(reference_top3[index])
            for index in range(38)
        )
    ),
    "maximumAbsoluteProbabilityDrift": float(
        np.abs(
            tfjs_probabilities - reference_probabilities
        ).max()
    ),
    "invalidProbabilityCases": invalid_probability_cases,
}
parity_report["releaseGatePassed"] = bool(
    parity_report["top1Matches"] == 38
    and parity_report["exactTop3Matches"] == 38
    and parity_report["top3SetMatches"] == 38
    and parity_report["maximumAbsoluteProbabilityDrift"] <= 1e-4
    and invalid_probability_cases == 0
)

PARITY_REPORT = FLOAT32_DIR / "parity_report.json"
PARITY_REPORT.write_text(
    json.dumps(parity_report, indent=2) + "\n"
)

print(json.dumps(parity_report, indent=2))
assert parity_report["releaseGatePassed"], (
    "Do not package this model: the TF.js parity gate failed."
)
print("Float32 TF.js model passed the final release gate.")


## 9. Package the frontend release

The release includes the model, five weight shards, labels, frontend metadata, the integration contract, parity evidence, and SHA-256 checksums.


In [ ]:
from IPython.display import FileLink, display

RELEASE_DIR = TFJS_ROOT / "agrieedge-v2-release"
ZIP_PATH = Path("/kaggle/working/AgriEdge-TFJS-v2-release.zip")

if RELEASE_DIR.exists():
    backup = RELEASE_DIR.with_name(
        "agrieedge-v2-release-backup-"
        + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    )
    shutil.move(str(RELEASE_DIR), str(backup))
    print("Earlier release preserved:", backup)

RELEASE_DIR.mkdir(parents=True, exist_ok=False)

for filename in ["model.json", "labels.json", *shard_names]:
    source = FLOAT32_DIR / filename
    assert source.is_file(), source
    shutil.copy2(source, RELEASE_DIR / filename)

shutil.copy2(PARITY_REPORT, RELEASE_DIR / "parity_report.json")

labels = json.loads((RELEASE_DIR / "labels.json").read_text())
assert isinstance(labels, list) and len(labels) == 38
assert [item["id"] for item in labels] == list(range(38))


def find_layers(value):
    if isinstance(value, dict):
        if isinstance(value.get("layers"), list):
            return value["layers"]
        for child in value.values():
            result = find_layers(child)
            if result is not None:
                return result
    elif isinstance(value, list):
        for child in value:
            result = find_layers(child)
            if result is not None:
                return result
    return None


layers = find_layers(model_document["modelTopology"])
assert layers is not None
layer_names = [
    layer.get("config", {}).get("name") for layer in layers
]
assert "Conv_1" in layer_names

created_at = datetime.now(timezone.utc).isoformat()

metadata_document = {
    "schemaVersion": 1,
    "releaseName": "AgriEdge MobileNetV2 v2",
    "modelVersion": "2.0.0",
    "createdAtUTC": created_at,
    "status": "release-approved",
    "modelType": "tfjs_layers_model",
    "architecture": "MobileNetV2",
    "classCount": 38,
    "modelFile": "model.json",
    "labelsFile": "labels.json",
    "input": {
        "name": "image",
        "shape": [1, 224, 224, 3],
        "dtype": "float32",
        "colorOrder": "RGB",
        "pixelRange": [0, 255],
        "normalizationEmbeddedInModel": True,
        "normalization": "Rescaling(1/127.5, offset=-1)",
        "frontendMustDivideBy255": False,
    },
    "output": {
        "name": "probabilities",
        "shape": [1, 38],
        "meaning": "Softmax probabilities ordered by labels.json id",
        "recommendedTopK": 3,
    },
    "decisionPolicy": {
        "minimumConfidence": 0.6,
        "minimumTop1Top2Margin": 0.1,
        "rejectedStatus": "uncertain",
        "userMessage": (
            "Prediction is uncertain. Capture another clear leaf image."
        ),
    },
    "runtime": {
        "preferredBackend": "webgl",
        "fallbackBackends": ["wasm", "cpu"],
        "offlineCompatible": True,
    },
    "gradCam": {
        "supported": True,
        "lastConvolutionLayer": "Conv_1",
    },
    "storageCompression": {
        "releasedFormat": "Float32",
        "quantization": "none",
        "rawBundleMiB": parity_report["rawBundleMiB"],
        "rationale": (
            "UINT8 was rejected after parity testing. Float32 "
            "achieved 38/38 Top-1 and Top-3 parity while remaining "
            "below the 10 MB deployable limit."
        ),
    },
    "parity": {
        **parity_report,
        "passed": True,
    },
}

integration_contract = {
    "modelUrl": "./model.json",
    "labelsUrl": "./labels.json",
    "metadataUrl": "./metadata.json",
    "imageWidth": 224,
    "imageHeight": 224,
    "channels": 3,
    "inputPixelMinimum": 0,
    "inputPixelMaximum": 255,
    "dividePixelsBy255": False,
    "topK": 3,
    "minimumConfidence": 0.6,
    "minimumMargin": 0.1,
    "lastConvolutionLayer": "Conv_1",
}

(RELEASE_DIR / "metadata.json").write_text(
    json.dumps(metadata_document, indent=2) + "\n"
)
(RELEASE_DIR / "integration_contract.json").write_text(
    json.dumps(integration_contract, indent=2) + "\n"
)

deployable_names = [
    "model.json",
    *shard_names,
    "labels.json",
    "metadata.json",
    "integration_contract.json",
]
deployable_paths = [RELEASE_DIR / name for name in deployable_names]
deployable_bytes = sum(path.stat().st_size for path in deployable_paths)
deployable_mb = deployable_bytes / 1_000_000
assert deployable_bytes < 10_000_000

manifest_files = sorted(
    path
    for path in RELEASE_DIR.iterdir()
    if path.is_file() and path.name != "artifact_manifest.json"
)
artifact_manifest = {
    "schemaVersion": 1,
    "releaseName": "agrieedge-v2-release",
    "createdAtUTC": created_at,
    "releaseApproved": True,
    "deployableBytes": deployable_bytes,
    "deployableMegabytes": round(deployable_mb, 3),
    "files": [
        {
            "name": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in manifest_files
    ],
}
(RELEASE_DIR / "artifact_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2) + "\n"
)

if ZIP_PATH.exists():
    zip_backup = ZIP_PATH.with_name(
        ZIP_PATH.stem
        + "-backup-"
        + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
        + ZIP_PATH.suffix
    )
    shutil.move(str(ZIP_PATH), str(zip_backup))

created_zip = Path(
    shutil.make_archive(
        str(ZIP_PATH.with_suffix("")),
        "zip",
        root_dir=str(RELEASE_DIR.parent),
        base_dir=RELEASE_DIR.name,
    )
)
assert created_zip == ZIP_PATH and ZIP_PATH.is_file()

print("Release directory:", RELEASE_DIR)
print("Deployable size:", round(deployable_mb, 3), "MB")
print("Weight shards:", len(shard_names))
print("Classes:", len(labels))
print("Top-1 parity: 38/38")
print("Exact Top-3 parity: 38/38")
print("Release gate passed: True")
print("Downloadable ZIP:", ZIP_PATH)

display(FileLink(str(ZIP_PATH)))


## Engineering decision: UINT8 was rejected

The archived notebook records a UINT8 converter experiment. Although that bundle deserialized, it matched only **15/38 Top-1** and **1/38 Top-3** golden cases, with probability drift close to `1.0`. It is not a deployable model.

Float32 was selected because it achieved strict **38/38 Top-1 and exact Top-3 parity**, remained below the 10 MB project limit, and eliminated a serious silent-correctness risk. Do not reintroduce the repaired UINT8 artifacts into the frontend.


## Frontend handoff

Extract `AgriEdge-TFJS-v2-release.zip` into:

```text
client/public/models/agrieedge-v2/
```

Frontend rules:

- resize to 224×224 RGB;
- pass pixel values in `[0, 255]`—**do not divide by 255**;
- display Top-3 results;
- mark the result uncertain when confidence `< 0.60` or the Top-1/Top-2 margin `< 0.10`;
- read `integration_contract.json` rather than duplicating constants;
- verify copied files against `artifact_manifest.json`.

Preserve this notebook, the archived debugging notebook, and the locked Python model outside the frontend bundle.
